# What is a perfect PV forecast worth to a home battery?

`CODE_PV_FORECAST.ipynb` ranks the forecasts a household could actually install.
`CODE_PV_PERFECT.ipynb` re-runs the same study with the generation channel replaced by the
realised roof. This notebook is the **difference between the two**, and it is the only one of
the three that is a measurement rather than a ranking.

### The pairing

Every row below is one **pair of arms** that differ in the generation channel and in nothing
else — same household, same 10 kWh battery, same tariff, same 24 h horizon, same load model,
same evaluator, and the *same forecast-blind oracle read out of the same cache entry*. The
oracle and every rule-based controller never look at a forecast, so `oracle_config` strips
the forecast axes out of their key and one solve serves both halves of a pair. That is what
makes the per-household difference the value of a perfect PV forecast rather than a
difference between two studies.

| load model | forecast roof | perfect roof |
|---|---|---|
| Prophet | `*_H24` | `*_H24_pvtruth` |
| yesterday | `*_H24_persist` | `*_H24_persist_pvtruth` |
| median of 14 d | `*_H24_median14` | `*_H24_median14_pvtruth` |
| tuned Prophet | `*_H24_prophet_tuned` | `*_H24_pvtruth_tuned` |
| median of 14 d + AR | `*_H24_hbd_median14` | `*_H24_hbd_median14_pvtruth` |

`pv_split.pv_pairs` builds that table from `hems_study.HYBRID_KINDS`, so a load model whose
perfect twin was never swept (`hbd`, `hbd_baseline`) is simply absent rather than silently
paired with a neighbouring model.

### The ladder

A pair is a **step** — forecast roof to perfect roof — and says nothing about what lies
between. So the second reading holds the load channel at Prophet and moves only the roof,
from copying yesterday up to knowing it:

`pvnaive` (roof = yesterday) → `pvmedian14` (roof = a 14-day median) → `H24` (roof = Prophet)
→ `pvtruth` (roof = the truth)

Those two intermediate arms exist for this figure and for nothing else, which is why they sit
in neither track and are swept **here**.

### Which way is up

Everything is stated as **what the perfect roof BUYS**: positive is the perfect-PV arm being
cheaper, or capturing more of the achievable gain. The headline unit is
`regret_pct_of_gain` — the share of what perfect foresight wins on that household that the
forecast threw away — because it is unit-free and it is the only form in which an AUD result
and a EUR result are the same quantity. The money is printed beside it, per tariff, never
across one.

### What this notebook owns

It sweeps the four `mixed` arms of the ladder and **reads** the other 31 from the checkpoints
the other two notebooks wrote. If an arm it needs is missing, the cell below says which
notebook owns it rather than recomputing someone else's sweep. `pv_split.__doc__` is the
cache contract in full.


In [ ]:
### Imports
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import stats as st

import hems_study as hs
hs = importlib.reload(hs)
import pv_split as ps
ps = importlib.reload(ps)

print(f"hems_study loaded: {len(hs.STUDY_ARMS)} arms in the roster.")
print("pv_split loaded  : "
      + ", ".join(f"{t} ({len(ps.arm_names(t))})" for t in ps.TRACKS)
      + " -- every arm in exactly one notebook.")


## 1. Configuration

This notebook compares **across** the partition, so unlike its two siblings it reads every
arm. It still only *writes* its own: the four ladder rungs, which belong to neither track.


In [ ]:
### What this notebook sweeps, and what it only reads
TRACK = ps.MIXED                       # the ladder rungs -- the only arms written here
RESULTS_DIR = hs.RESULTS_DIR
SUBDIR = ps.FIGURE_SUBDIR[TRACK]
RUN_SWEEP = True                       # resumable; a warm checkpoint is read, not re-solved
N_SIM = 365
N_JOBS = 10

# OWNED vs READ, and the distinction is the whole of the concurrency story. The
# three notebooks partition `hs.STUDY_ARMS`, so two of them can run at once and
# never write one checkpoint -- while still sharing every content-keyed cache
# under `forecast_cache/`, `oracle_cache/` and `hbd_params/`. Sweeping the arms
# below without `arms=OWN_SPECS` would re-run the other notebooks' arms from
# here, which is the one way to get two kernels onto one checkpoint.
OWN_SPECS = ps.arm_specs(TRACK)
OWN = ps.arm_names(TRACK)
READS = {t: ps.arm_names(t) for t in (ps.FORECAST, ps.PERFECT)}
ARMS = [a["name"] for a in hs.STUDY_ARMS]

# The comparison itself, built from the arm definitions rather than listed. A
# pair is (load model, forecast-roof arm, perfect-roof arm); the ladder pins the
# load model and walks the roof from yesterday to the truth.
TARIFFS = sorted(hs.REFERENCE_ARM)
PAIRS = {t: ps.pv_pairs(t) for t in TARIFFS}
LADDER = {t: ps.roof_ladder(t) for t in TARIFFS}

print(f"sweeps  : {len(OWN)} arm(s) -- {', '.join(OWN)}")
for _t, _names in READS.items():
    print(f"reads   : {len(_names):2d} arm(s) from the {_t} notebook")
print(f"figures : Results/Figures/{SUBDIR}/")

print(f"\npairs   : the generation channel, and nothing else, differs within a row")
_w = max(len(hs.forecast_kind_label(k)) for t in TARIFFS for k, _, _ in PAIRS[t])
for _t in TARIFFS:
    for _kind, _base, _truth in PAIRS[_t]:
        print(f"   {_t}  {ps.source_label(_kind):{_w}s}  {_base:26s} -> {_truth}")

print(f"\nladder  : the load channel pinned, the roof walking up")
for _t in TARIFFS:
    print(f"   {_t}  " + "  ->  ".join(f"{ps.source_label(g)} ({a})"
                                       for g, a in LADDER[_t]))


## 2. Run or load

The four ladder rungs are swept here; the other 31 arms are read from the checkpoints the
other two notebooks wrote. A missing arm is named with the notebook that owns it — this one
does not recompute another notebook's sweep, because two kernels writing one checkpoint is
how a sweep loses a result rather than a race.


In [ ]:
### Sweep the ladder, read everything else
if RUN_SWEEP:
    _swept = hs.run_arms(
        data_dir=str(Path("..") / "Input data" / "Ausgrid"),
        output_root=str(RESULTS_DIR),
        dataset_ids=hs.dataset_ids(),
        n_sim=N_SIM,
        n_jobs=N_JOBS,
        # THE LADDER RUNGS AND NOTHING ELSE -- see the note in the cell above.
        arms=OWN_SPECS,
    )

df_all = hs.collect_results(RESULTS_DIR, arms=set(ARMS))
if df_all.empty:
    raise RuntimeError(f"No results in {RESULTS_DIR}. Run CODE_PV_FORECAST.ipynb first.")
df_all = df_all[df_all["roster_complete"]].reset_index(drop=True)

# WHOSE SWEEP IS MISSING, said by name. This notebook compares arms it does not
# own, so "an arm is absent" has a specific remedy -- run the notebook that owns
# it -- and a generic "no results" would send someone to re-sweep the lot from
# here, which is exactly the thing the partition exists to prevent.
_have = set(df_all["arm"])
_missing = {t: [a for a in names if a not in _have] for t, names in READS.items()}
_missing = {t: a for t, a in _missing.items() if a}
if _missing:
    _owner = {ps.FORECAST: "CODE_PV_FORECAST.ipynb", ps.PERFECT: "CODE_PV_PERFECT.ipynb"}
    raise RuntimeError(
        "This notebook compares arms it does not sweep, and some have not run:\n"
        + "\n".join(f"  {_owner[t]} owns {len(a)} missing arm(s): {', '.join(a)}"
                    for t, a in _missing.items())
        + "\nRun that notebook's sweep cell. Do NOT sweep them from here -- the "
          "three notebooks partition the roster so that two kernels never write "
          "one checkpoint."
    )

# A household enters only if it finished EVERY arm, on both sides of every pair.
# The pairing is the unit of analysis here, so a household present on one half of
# a pair and absent on the other would turn a paired difference into a difference
# of two samples.
_done = df_all.groupby("dataset")["arm"].nunique()
_balanced = set(_done[_done == len(_have)].index)
_held = set(df_all["dataset"]) - _balanced
if _held:
    print(f"Held out to keep every pair paired: {len(_held)} household(s) -- "
          f"{', '.join(sorted(_held))}")
df_all = df_all[df_all["dataset"].isin(_balanced)].reset_index(drop=True)

long = hs.summarize(df_all)
print(f"Comparing {df_all['dataset'].nunique()} households across "
      f"{len(_have)} arms = {len(df_all)} runs.")
print(f"   {len(OWN)} swept here, {len(_have) - len(OWN)} read from the other two notebooks.")

# THE PAIRS ARE A CONTROLLED COMPARISON, and this is the check that they are.
# The oracle and every rule are forecast-blind, so `oracle_config` drops the
# forecast axes from their cache key and both halves of a pair are served the
# SAME solve. If that ever stopped being true, the difference below would carry a
# dispatch difference the forecast did not cause -- so it is verified rather than
# asserted, on the two columns that would move first.
_probe = df_all.set_index(["arm", "dataset"])
_bad = []
for _t in TARIFFS:
    for _kind, _base, _truth in PAIRS[_t]:
        for _col in ("cost_oracle", "cost_no_battery", "cost_milp_full"):
            _d = (_probe.loc[_base, _col] - _probe.loc[_truth, _col]).abs().max()
            if _d > 1e-6:
                _bad.append(f"{_base} vs {_truth}: {_col} differs by up to {_d:.2e}")
print(f"\nforecast-blind controls identical across all {sum(len(PAIRS[t]) for t in TARIFFS)}"
      f" pairs: {'yes' if not _bad else 'NO -- ' + '; '.join(_bad)}")


## 3. What the perfect roof buys, per load model

One row per (tariff, load model): the same MPC, the same everything, with the generation
channel swapped for the truth.

`Regret closed` is the headline. `regret_pct_of_gain` is the share of what perfect foresight
wins on a household that a forecast threw away, and both halves of a pair divide by the *same*
denominator — the oracle and the no-battery reference are forecast-blind, so the gain is a
property of the household, not of the arm. The difference is therefore exactly the share of
the achievable gain that the roof channel was costing, per household, before any median.

`of its regret` is the same number as a fraction of what that load model was throwing away in
the first place: 100 % would mean the roof channel was the *entire* forecast penalty.

The test is **Wilcoxon signed-rank on the per-household differences**, not a comparison of two
medians: the households are paired by construction and a difference of medians throws that
away. `better` counts the households the perfect roof helped.


In [ ]:
### The value of the sun, per load model and tariff
def pv_effect(tariff, base_arm, truth_arm, controller="prophet"):
    """Per household, what replacing the roof forecast with the truth did.

    Returns a frame indexed by household. Sign convention throughout this
    notebook: POSITIVE IS WHAT THE PERFECT ROOF BUYS. `money` is the base arm's
    total cost minus the perfect arm's, so a cheaper perfect arm is positive;
    `regret_closed` is the base arm's regret minus the perfect arm's, so a
    smaller regret under perfect PV is positive. Getting this backwards is the
    one error the whole notebook would be invisible to, which is why it is stated
    here rather than at each call.
    """
    cols = ["cost", "cost_total", "regret_pct_of_gain", "efc", "npv",
            "saving_total_pct"]
    b = (long[(long["arm"] == base_arm) & (long["controller"] == controller)]
         .set_index("dataset")[cols])
    t = (long[(long["arm"] == truth_arm) & (long["controller"] == controller)]
         .set_index("dataset")[cols])
    hh = b.index.intersection(t.index)
    b, t = b.loc[hh], t.loc[hh]
    return pd.DataFrame({
        "money":         b["cost_total"] - t["cost_total"],
        "bill":          b["cost"] - t["cost"],
        "regret_base":   b["regret_pct_of_gain"],
        "regret_truth":  t["regret_pct_of_gain"],
        "regret_closed": b["regret_pct_of_gain"] - t["regret_pct_of_gain"],
        "efc":           t["efc"] - b["efc"],
        "npv":           t["npv"] - b["npv"],
    })


def signed_rank(diffs):
    """Wilcoxon on the paired differences, with the degenerate cases named.

    `scipy` raises on an all-zero difference vector rather than reporting the
    obvious answer, and an all-zero vector is exactly what a pair whose forecast
    never mattered would produce -- i.e. the result this notebook exists to be
    able to report. NaN, and the caller prints `n/a`.
    """
    d = np.asarray([x for x in diffs if x == x], dtype=float)
    if len(d) < 2 or np.allclose(d, 0.0):
        return float("nan")
    return float(st.wilcoxon(d).pvalue)


EFFECT = {}
for tariff in TARIFFS:
    cur = hs.currency(hs.REFERENCE_ARM[tariff])
    rows = []
    for kind, base, truth in PAIRS[tariff]:
        e = pv_effect(tariff, base, truth)
        EFFECT[(tariff, kind)] = e
        rows.append({
            "load model": ps.source_label(kind),
            "regret_base": e["regret_base"].median(),
            "regret_truth": e["regret_truth"].median(),
            "closed": e["regret_closed"].median(),
            "of_regret": 100.0 * e["regret_closed"].median() / e["regret_base"].median()
                         if e["regret_base"].median() > 1e-9 else float("nan"),
            "money": e["money"].median(),
            "efc": e["efc"].median(),
            "better": int((e["regret_closed"] > 0).sum()),
            "n": int(e["regret_closed"].notna().sum()),
            "p": signed_rank(e["regret_closed"]),
        })
    tab = pd.DataFrame(rows)
    _w = max(len(r) for r in tab["load model"]) + 2
    print(f"=== {tariff}: what a perfect roof buys the MPC, money in {cur} ===")
    print(f"{'load model':{_w}s}{'regret':>8s}{'-> ':>4s}{'':>7s}"
          f"{'closed':>8s}{'of its':>8s}{cur + '/a':>9s}{'EFC/a':>8s}"
          f"{'better':>8s}{'p':>9s}")
    print(f"{'':{_w}s}{'%gain':>8s}{'':>4s}{'%gain':>7s}"
          f"{'pts':>8s}{'regret':>8s}{'':>9s}{'':>8s}{'of ' + str(tab['n'].max()):>8s}")
    for _, r in tab.iterrows():
        p = "n/a" if r["p"] != r["p"] else f"{r['p']:.4f}"
        print(f"{r['load model']:{_w}s}{r['regret_base']:8.1f}{'->':>4s}"
              f"{r['regret_truth']:7.1f}{r['closed']:8.1f}{r['of_regret']:7.0f}%"
              f"{r['money']:9.2f}{r['efc']:8.1f}{r['better']:8d}{p:>9s}")
    print()

print("Positive is what the perfect roof BUYS: a smaller regret, a cheaper year.")
print("`closed` is in POINTS of the achievable gain and `of its regret` is that as a share")
print("of what the load model was throwing away -- 100 % would mean the roof channel was the")
print("entire forecast penalty. `EFC/a` is the change in cycles, i.e. whether knowing the sun")
print("changes what the battery DOES and not only what it costs.")


## 4. Plots

Colour is the tariff — AU blue, SI orange — exactly as in the other two notebooks, so a
reader who has learnt the palette there does not have to relearn it here. Every figure goes
through `Plotting_Functions`, which decides style, size and export, and writes the caption and
provenance beside each figure under `Results/Figures/hems_pv_value/`.


In [ ]:
### Shared chart styling
import importlib
import textwrap

import Plotting_Functions as pf
pf = importlib.reload(pf)
pf.use(subdir=SUBDIR, titles=False)
from Plotting_Functions import INK, INK_2, MUTED, SURFACE, SERIES, finish

import figure_style as fs
fs = importlib.reload(fs)
from figure_style import (TARIFF_COLOR, FAMILY_MARKER, BEATS, LOSES,
                          beeswarm, clip_window, boot_ci, rho_lines,
                          corner_block, PCT_OF_GAIN_SHORT)

SAMPLE = f"{df_all['dataset'].nunique()} households, one per k-means cluster"
# ROW ORDER, FIXED ONCE. Every figure below has the load models on an axis, and
# they have to be in the same order on all of them or the eye cannot carry a row
# from one figure to the next. `FORECAST_KIND_LABELS` is the roster's own order
# and both tariffs pair the same five models, so one list serves.
LOADS = [k for k, _, _ in PAIRS[TARIFFS[0]]]
LOAD_LABEL = {k: ps.source_label(k) for k in LOADS}
print(f"Style ready: {SAMPLE}, {len(LOADS)} paired load models, "
      f"semantics from figure_style.py.")


In [ ]:
### Figure: what a perfect roof buys each load model
# ONE DOT PER HOUSEHOLD, and the bar is the paired median. The median of the
# per-household DIFFERENCES is the statistic -- the households are paired by
# construction -- and a bar alone would hide that on most rows the effect is
# small and on a few it is not, which is the finding.
#
# DRAWN ON `regret_closed`, in POINTS OF THE ACHIEVABLE GAIN. That is the only
# form in which the AU panel and the SI panel are the same quantity: the money
# version is AUD on one and EUR on the other, over different bills, and the two
# panels could then only be read separately. The money is in the table above.
#
# ONE X SCALE ACROSS BOTH PANELS, for the same reason: "is the sun worth more on
# SI than on AU" has to be readable off the bar lengths.
from matplotlib.lines import Line2D

_pool = [v for t in TARIFFS for k in LOADS
         for v in EFFECT[(t, k)]["regret_closed"].dropna()]
_meds = [float(EFFECT[(t, k)]["regret_closed"].median())
         for t in TARIFFS for k in LOADS]
_win = clip_window(_pool, _meds, q=0.02, pad=0.10)

fig, axes = plt.subplots(1, len(TARIFFS), figsize=pf.figsize(ratio=0.58),
                         sharey=True, sharex=True)
axes = np.atleast_1d(axes)
_clipped = 0
for ax, tariff in zip(axes, TARIFFS):
    col = TARIFF_COLOR[tariff]
    y = np.arange(len(LOADS))[::-1]
    for yi, kind in zip(y, LOADS):
        v = EFFECT[(tariff, kind)]["regret_closed"].dropna().to_numpy()
        med = float(np.median(v)) if len(v) else float("nan")
        # CLIPPED, NOT DROPPED: anything outside the window is drawn on the
        # boundary as a sideways caret and counted in the key, so a reader is
        # told what is off the edge rather than shown a tidier distribution.
        vv = np.clip(v, _win[0], _win[1]) if _win else v
        _clipped += int(np.sum(vv != v))
        ax.scatter(vv, yi + beeswarm(vv), s=13, color=col, alpha=0.45,
                   linewidths=0, zorder=2)
        ax.plot([0, med], [yi, yi], color=col, lw=3.2, solid_capstyle="butt",
                alpha=0.85, zorder=3)
        ax.scatter([med], [yi], s=70, marker=FAMILY_MARKER["MPC"], color=col,
                   edgecolor=SURFACE, lw=0.7, zorder=4)
        ax.annotate(f"{med:+.1f}", xy=(med, yi), xytext=(7 if med >= 0 else -7, 7),
                    textcoords="offset points", fontsize=7.5, color=INK_2,
                    ha="left" if med >= 0 else "right", va="center")
    ax.axvline(0, color=INK, lw=1.2, zorder=1)
    ax.set_yticks(y, [LOAD_LABEL[k] for k in LOADS])
    if _win:
        ax.set_xlim(*_win)
    ax.grid(axis="y", visible=False)
    # Panel identity in the X LABEL: `pf.use(titles=False)` lifts axes titles
    # into the export's caption, which would leave two unlabelled panels.
    finish(ax, xlabel=f"{tariff} — {PCT_OF_GAIN_SHORT} closed")

_key = [Line2D([], [], marker=FAMILY_MARKER["MPC"], ls="", ms=9,
               color=TARIFF_COLOR[t], label=f"{t} paired median") for t in TARIFFS]
_key.append(Line2D([], [], marker="o", ls="", ms=5, color=MUTED, alpha=0.5,
                   label="one household"))
pf.chart_frame(
    fig,
    "What a perfect PV forecast buys each load model, in points of the gain "
    "perfect foresight achieves on that household. Positive is the perfect roof "
    "winning; the bar is the median of the per-household differences, which is "
    "the paired statistic"
    + (f", and {_clipped} point(s) outside the window are drawn on the boundary"
       if _clipped else ""),
    handles=_key, ncol=3)
pf.show(fig, "pv_value_by_load_model", layout="frame")


### 4.1 The ladder: how good does the roof forecast have to be?

The table above is a *step* — a real forecast, then the truth — and a step cannot say whether
the value is in the last mile or the first. This holds the consumption channel at Prophet and
moves only the roof: copy yesterday, take a 14-day median, fit Prophet, then know it exactly.

If the ladder is flat between the first rung and the last, the roof channel is not where the
money is and a better PV model is wasted effort. If it climbs only at the top, the value is in
*perfect* knowledge specifically, which nothing deployable can reach.

**The rungs are not in measured-skill order, and the figure is drawn that way on purpose.**
They run from the crudest roof to the truth, with the study's own forecaster second from the
top — where it belongs in the roster, not where its error puts it. On the generation channel
Prophet scores **−0.17** against seasonal-naive while a 14-day median scores **+0.11**, so the
line is expected to *dip* at the Prophet rung. Sorting the rungs by skill would hide that
inside the axis; leaving it makes the roster's most uncomfortable measurement visible as a
shape rather than a footnote.


In [ ]:
### The roof-quality ladder, with the load channel pinned
# The rungs are ordered by the roof's own forecast skill, worst first, so the rung
# order IS the quality order -- see `ps.roof_ladder`. The load channel is Prophet
# on every rung by construction, which is what makes the roof the only thing
# moving; `pvnaive` and `pvmedian14` exist for this figure and nothing else.
LAD = {}
for tariff in TARIFFS:
    cur = hs.currency(hs.REFERENCE_ARM[tariff])
    rows = []
    for gen, arm in LADDER[tariff]:
        s = (long[(long["arm"] == arm) & (long["controller"] == "prophet")]
             .set_index("dataset"))
        LAD[(tariff, gen)] = s
        rows.append({"roof": ps.source_label(gen), "arm": arm,
                     "regret": s["regret_pct_of_gain"].median(),
                     "cost": s["cost_total"].median(),
                     "efc": s["efc"].median()})
    tab = pd.DataFrame(rows)
    # AGAINST THE BOTTOM RUNG, per household and then median'd. Against the
    # PREVIOUS rung would make each row's baseline a different arm and the column
    # would not sum to the ladder's height.
    _floor = LADDER[tariff][0][1]
    _f = (long[(long["arm"] == _floor) & (long["controller"] == "prophet")]
          .set_index("dataset")["regret_pct_of_gain"])
    tab["vs_floor"] = [
        float((_f - LAD[(tariff, gen)]["regret_pct_of_gain"]).median())
        for gen, _ in LADDER[tariff]]
    _w = max(len(r) for r in tab["roof"]) + 2
    print(f"=== {tariff}: the roof channel alone, load held at Prophet, money in {cur} ===")
    print(f"{'roof forecast':{_w}s}{'regret':>9s}{'closed':>9s}{cur + '/a':>10s}"
          f"{'EFC/a':>8s}  arm")
    print(f"{'':{_w}s}{'%gain':>9s}{'vs yday':>9s}{'total':>10s}{'':>8s}")
    for _, r in tab.iterrows():
        print(f"{r['roof']:{_w}s}{r['regret']:9.1f}{r['vs_floor']:9.1f}"
              f"{r['cost']:10.2f}{r['efc']:8.1f}  {r['arm']}")
    print()
print("`closed vs yday` is per household against the bottom rung, then median'd -- the height")
print("of the ladder up to that point. The top rung is the perfect roof, so its value is the")
print("whole of what the roof channel can ever be worth to a Prophet load model.")


In [ ]:
### Figure: the ladder
# A LINE, not bars. The rungs are ORDERED -- they are the same channel at four
# qualities -- so the question the figure has to answer is the SHAPE: does the
# value arrive gradually, or only at the top rung that nothing deployable can
# reach? Bars would put four independent-looking quantities side by side and
# leave that to be inferred.
#
# BOTH TARIFFS ON ONE AXES, because the y unit is a share of that household's own
# achievable gain and is therefore the same quantity on both. The colour is the
# tariff, as everywhere else in the three notebooks.
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=pf.figsize(ratio=0.50))
for tariff in TARIFFS:
    rungs = LADDER[tariff]
    x = np.arange(len(rungs))
    col = TARIFF_COLOR[tariff]
    _f = LAD[(tariff, rungs[0][0])]["regret_pct_of_gain"]
    med, lo, hi = [], [], []
    for gen, _ in rungs:
        d = (_f - LAD[(tariff, gen)]["regret_pct_of_gain"]).dropna()
        m = float(d.median())
        # The interval is a percentile bootstrap of the PAIRED median over the
        # same households -- n = 30 ratios with a few far out, which is not a
        # sample a normal approximation should be asked about.
        l, h = boot_ci(d.to_numpy())
        med.append(m); lo.append(m - l); hi.append(h - m)
    ax.errorbar(x, med, yerr=[lo, hi], color=col, lw=2.0, capsize=3,
                marker=FAMILY_MARKER["MPC"], ms=11, mec=SURFACE, mew=0.7,
                zorder=3, label=tariff)
    ax.annotate(f"{med[-1]:+.1f}", xy=(x[-1], med[-1]), xytext=(8, 0),
                textcoords="offset points", color=col, fontsize=8.5, va="center")
ax.axhline(0, color=INK, lw=1.2, zorder=1)
# The rung names are the same on both tariffs by construction -- the ladder pins
# the load model, and the roof sources are a property of the roster, not of a
# price signal. Taken from the first tariff and checked against the second.
_names = [ps.source_label(g) for g, _ in LADDER[TARIFFS[0]]]
assert all([ps.source_label(g) for g, _ in LADDER[t]] == _names for t in TARIFFS)
ax.set_xticks(np.arange(len(_names)), _names)
ax.grid(axis="x", visible=False)
finish(ax, xlabel="The roof forecast — consumption held at Prophet",
       ylabel=f"{PCT_OF_GAIN_SHORT} closed\nagainst a roof copied from yesterday")

pf.chart_frame(
    fig,
    "How much the roof channel is worth at four qualities, with the consumption "
    "channel held at Prophet. Each point is the paired median over "
    f"{df_all['dataset'].nunique()} households against the bottom rung, with a 95 % "
    "percentile-bootstrap interval; the top rung is the truth, so it is the whole "
    "of what a PV forecast can ever buy this load model",
    handles=[Line2D([], [], marker=FAMILY_MARKER["MPC"], ls="-", lw=2, ms=10,
                    color=TARIFF_COLOR[t], label=t) for t in TARIFFS],
    ncol=2)
pf.show(fig, "pv_roof_quality_ladder", layout="frame")


## 5. Does perfect PV change the *control* decision?

Everything above prices the forecast. This asks the question a household actually faces: with
a perfect roof in hand, does the controller **do** anything different, and would you **install**
anything different?

* **Cycles.** If the MPC spends the same pack life either way, the perfect roof bought a
  cheaper bill out of the same dispatch — it is a better *plan*, not a different *policy*.
* **The ranking.** Every figure in the other two notebooks ends in an ordering of controllers.
  If that ordering is the same under a perfect roof, then the recommendation the study makes
  does not depend on the PV forecast at all, which is a stronger statement than any of the
  medians above.


In [ ]:
### Does the perfect roof change the dispatch, or only the bill?
BASE_ARM = {(t, k): b for t in TARIFFS for k, b, _ in PAIRS[t]}
for tariff in TARIFFS:
    cur = hs.currency(hs.REFERENCE_ARM[tariff])
    print(f"=== {tariff}: the MPC's own behaviour, perfect roof minus forecast roof ===")
    _w = max(len(ps.source_label(k)) for k in LOADS) + 2
    print(f"{'load model':{_w}s}{'EFC/a':>9s}{'as %':>8s}{cur + '/a':>10s}"
          f"{'NPV':>10s}{'p(EFC)':>9s}")
    for kind in LOADS:
        e = EFFECT[(tariff, kind)]
        base_efc = (long[(long["arm"] == BASE_ARM[(tariff, kind)])
                         & (long["controller"] == "prophet")]["efc"].median())
        pct = 100.0 * e["efc"].median() / base_efc if base_efc > 1e-9 else float("nan")
        p = signed_rank(e["efc"])
        print(f"{ps.source_label(kind):{_w}s}{e['efc'].median():9.1f}{pct:7.1f}%"
              f"{e['money'].median():10.2f}{e['npv'].median():10.0f}"
              f"{('n/a' if p != p else f'{p:.4f}'):>9s}")
    print()
print("EFC is equivalent full cycles a year. A dispatch that barely moves while the bill does")
print("means the perfect roof bought a better PLAN on the same POLICY; a dispatch that moves")
print("means the controller would physically run the pack differently if it knew the sun.")


In [ ]:
### Would you install anything different? The controller ranking, both ways
# THE QUESTION THE MEDIANS CANNOT ANSWER. Every ranking figure in the other two
# notebooks orders controllers by `saving_total_pct` -- bill, standing charge and
# wear, the quantity the whole-period solve minimises and the only one it bounds.
# If that ORDER survives a perfect roof, then the study's recommendation does not
# depend on the PV forecast, which is a stronger claim than any euro figure here.
#
# KENDALL'S TAU-B, not Spearman: this is a comparison of two ORDERINGS over a
# dozen controllers, where what matters is how many pairs of controllers swap,
# and tau is the statistic that counts exactly that. Ties are possible -- the
# rules are byte-identical across a pair, since no rule reads a forecast -- so
# tau-b, which corrects for them.
# WHAT "BEST" MEANS HERE. `milp_full` wins every ordering by construction -- it
# is one solve over the whole scored year and the bound the rest are measured
# against -- and `oracle` and `price_oracle` read data no controller has. Printed
# as "the best controller" any of the three is a recommendation to install
# something that does not exist. The TAU still runs over every controller,
# because a reordering anywhere is a reordering; only the headline is restricted.
NOT_DEPLOYABLE = ("no_battery", "milp_full", "oracle", "price_oracle")

for tariff in TARIFFS:
    base_arm, truth_arm = hs.REFERENCE_ARM[tariff], ps.reference_arm(ps.PERFECT)[tariff]
    def _order(arm, deployable_only=False):
        drop = NOT_DEPLOYABLE if deployable_only else ("no_battery",)
        s = (long[(long["arm"] == arm) & (~long["controller"].isin(drop))]
             .groupby("controller")["saving_total_pct"].median())
        return s.reindex([c for c in hs.controller_columns(df_all) if c in s.index])
    a, b = _order(base_arm), _order(truth_arm)
    common = [c for c in a.index if c in b.index]
    a, b = a.loc[common], b.loc[common]
    tau = st.kendalltau(a.rank(), b.rank())
    ra, rb = a.rank(ascending=False), b.rank(ascending=False)
    moved = [(c, int(ra[c]), int(rb[c])) for c in common if ra[c] != rb[c]]
    da, db = _order(base_arm, True), _order(truth_arm, True)
    print(f"=== {tariff}: {base_arm} vs {truth_arm} ===")
    print(f"   best DEPLOYABLE   forecast roof: {hs.controller_label(da.idxmax(), 48, 0.5, N_SIM)}")
    print(f"                     perfect  roof: {hs.controller_label(db.idxmax(), 48, 0.5, N_SIM)}")
    print(f"   Kendall tau-b over {len(common)} controllers: {tau.statistic:+.3f} "
          f"(p = {tau.pvalue:.2g})")
    if moved:
        print(f"   {len(moved)} controller(s) change rank:")
        for c, i, j in sorted(moved, key=lambda r: r[1]):
            print(f"      {hs.controller_label(c, 48, 0.5, N_SIM):<48s} {i} -> {j}")
    else:
        print("   no controller changes rank: the perfect roof reorders nothing.")
    # AND THE ONE RANK A READER ACTS ON. A tau of 0.95 can still hide a swap at
    # the top, which is the only position anyone installs from.
    print(f"   the MPC's own rank: {int(ra['prophet'])} -> {int(rb['prophet'])} "
          f"of {len(common)}")
    print()


## 6. Which households is the sun worth something to?

Every number above is a median over 30 households. These ask what a median cannot: is the
value of a perfect roof concentrated in a few sites, and do those sites have anything in
common — more surplus PV, or a roof that the forecast happens to be bad at?

The second is the sharper test. If the households where perfect PV pays are the households
whose PV forecast has the worst *error*, then the money is in the forecast and a better model
would collect it. If there is no such relationship, then the error the roof channel makes is
not the error that costs anything, and the two are measuring different things.


In [ ]:
### Which households, and is the answer a gradient?
# DRAWN IN THE SPACE THE TEST MEASURES. Spearman is a rank statistic and knows
# nothing about the spacing of its inputs, so x is the RANK, 1 to n. Every
# household then carries the same visual weight it has in the statistic, and the
# monotone trend the coefficient describes is the trend the eye sees.
#
# THE REFERENCE PAIR ONLY. Five load models would be five panels of the same
# shape; the study's own forecaster is the one the paper is about, and the table
# in section 3 already says whether the others behave differently.
_HET = ["sell_no_battery", "buy_no_battery", "gen_skill_vs_naive"]
het = {}
for tariff in TARIFFS:
    base_arm = hs.REFERENCE_ARM[tariff]
    e = EFFECT[(tariff, ps.arm_forecast_kind(base_arm))]
    side = (df_all[df_all["arm"] == base_arm].set_index("dataset")[_HET]
            .reindex(e.index))
    het[tariff] = e.join(side).assign(
        pv_ratio=lambda d: 100.0 * d["sell_no_battery"] / d["buy_no_battery"])

fig, axes = plt.subplots(len(TARIFFS), 2, figsize=pf.figsize(ratio=0.80),
                         sharey="row", sharex="col")
axes = np.atleast_2d(axes)
for row, tariff in enumerate(TARIFFS):
    s = het[tariff]
    col = TARIFF_COLOR[tariff]
    for ax, xcol, what in zip(axes[row],
                              ("pv_ratio", "gen_skill_vs_naive"),
                              ("surplus PV (rank, least to most)",
                               "the roof forecast's own skill (rank, worst to best)")):
        d = s[[xcol, "regret_closed"]].dropna()
        xr = st.rankdata(d[xcol].to_numpy())
        y = d["regret_closed"].to_numpy()
        ax.scatter(xr, y, s=58, marker=FAMILY_MARKER["MPC"], color=col,
                   edgecolor=SURFACE, lw=0.7, alpha=0.9, zorder=3)
        # THEIL-SEN IN RANK SPACE: the median of the pairwise slopes, which is
        # the rank-based line a rank-based coefficient describes. Not OLS, which
        # would be a claim about magnitudes in a plane that has none.
        if len(d) > 2:
            sl, ic, _, _ = st.theilslopes(y, xr)
            xx = np.array([xr.min(), xr.max()])
            ax.plot(xx, ic + sl * xx, color=col, lw=1.4, ls="--", alpha=0.8, zorder=2)
            # RHO AND ITS INTERVAL FROM ONE CALL. `fs.spearman_ci` returns
            # (rho, p, lo, hi, n) -- taking rho from `st.spearmanr` and the
            # bounds from here would be two estimates of the same thing, and
            # they are bootstrapped over the same draw for a reason.
            rho, p, lo, hi, n = fs.spearman_ci(xr, y)
            corner_block(ax, rho_lines(tariff, rho, p, lo, hi, n), col, y)
        ax.axhline(0, color=INK, lw=1.0, zorder=1)
        finish(ax, xlabel=what if row == len(TARIFFS) - 1 else None,
               ylabel=f"{tariff} — {PCT_OF_GAIN_SHORT}\nclosed by a perfect roof"
                      if ax is axes[row][0] else None)

pf.chart_frame(
    fig,
    "Which households a perfect PV forecast is worth something to. Each mark is "
    f"one of {df_all['dataset'].nunique()} households under the study's own "
    "forecaster; y is the share of that household's achievable gain the perfect "
    "roof recovers. Left: against how much surplus PV the site exports. Right: "
    "against how well the roof forecast scored on that site, so a positive trend "
    "would mean the money is where the forecast error is",
    handles=[Line2D([], [], marker=FAMILY_MARKER["MPC"], ls="", ms=10,
                    color=TARIFF_COLOR[t], label=t) for t in TARIFFS],
    ncol=2)
pf.show(fig, "pv_value_heterogeneity", layout="frame")


## 7. Read-out

The three sentences the notebook exists to be able to write, computed rather than typed.


In [ ]:
### The answer, stated
_all = pd.concat([EFFECT[(t, k)].assign(tariff=t, kind=k)
                  for t in TARIFFS for k in LOADS], ignore_index=False)
print("1. WHAT THE SUN IS WORTH")
for tariff in TARIFFS:
    cur = hs.currency(hs.REFERENCE_ARM[tariff])
    sub = _all[_all["tariff"] == tariff]
    m = sub["regret_closed"].median()
    lo, hi = boot_ci(sub["regret_closed"].dropna().to_numpy())
    print(f"   {tariff}: pooled over {len(LOADS)} load models and "
          f"{sub.index.nunique()} households, a perfect roof recovers a median "
          f"{m:+.1f} points")
    print(f"       of the achievable gain (95 % CI [{lo:+.1f}, {hi:+.1f}]), worth "
          f"{sub['money'].median():+.2f} {cur}/a.")

print("\n2. WHERE ON THE LADDER IT ARRIVES")
for tariff in TARIFFS:
    rungs = LADDER[tariff]
    _f = LAD[(tariff, rungs[0][0])]["regret_pct_of_gain"]
    steps = [(ps.source_label(g),
              float((_f - LAD[(tariff, g)]["regret_pct_of_gain"]).median()))
             for g, _ in rungs]
    top = steps[-1][1]
    # WHAT A DEPLOYABLE ROOF ALREADY COLLECTS, as a share of the whole ladder.
    # The interesting number is not the top of the ladder -- nothing reaches it --
    # but how much of it the best rung a household can actually install is worth.
    best_dep = max(steps[:-1], key=lambda s: s[1])
    share = 100.0 * best_dep[1] / top if abs(top) > 1e-9 else float("nan")
    print(f"   {tariff}: the whole ladder is {top:+.1f} points; the best DEPLOYABLE "
          f"roof ({best_dep[0]})")
    print(f"       is worth {best_dep[1]:+.1f}, i.e. {share:.0f} % of it. "
          + "  ".join(f"{n} {v:+.1f}" for n, v in steps))

print("\n3. WHETHER IT CHANGES THE DECISION")
for tariff in TARIFFS:
    base_arm, truth_arm = hs.REFERENCE_ARM[tariff], ps.reference_arm(ps.PERFECT)[tariff]
    def _best(arm):
        # DEPLOYABLE ONLY -- `milp_full` wins every arm by construction and
        # naming it here would read as a purchase recommendation for the bound.
        s = (long[(long["arm"] == arm)
                  & (~long["controller"].isin(
                      ("no_battery", "milp_full", "oracle", "price_oracle")))]
             .groupby("controller")["saving_total_pct"].median())
        return s.idxmax()
    same = _best(base_arm) == _best(truth_arm)
    e = EFFECT[(tariff, ps.arm_forecast_kind(base_arm))]
    print(f"   {tariff}: the best deployable controller is "
          f"{'the SAME' if same else 'DIFFERENT'} under a perfect roof "
          f"({hs.controller_label(_best(base_arm), 48, 0.5, N_SIM)}"
          + ("" if same else f" -> {hs.controller_label(_best(truth_arm), 48, 0.5, N_SIM)}")
          + ").")
    print(f"       The MPC's own cycling moves {e['efc'].median():+.1f} EFC/a "
          f"and its lifetime NPV {e['npv'].median():+.0f} "
          f"{hs.currency(base_arm)}.")


### Caveats

* **`TruthForecaster` is day-ahead, not omniscient.** The table it serves is anchored at the
  start of a local day and holds exactly that day, so a controller reading it knows the next
  24 h of generation perfectly and nothing beyond. That is deliberately the *same* information
  boundary the forecast arms have — which is what makes the pair a clean comparison — but it
  means the numbers here bound a perfect **day-ahead** PV forecast, not perfect foresight.
  `oracle`, which re-reads the truth at every step on both channels, is the other bound and is
  in the two track notebooks.

* **The load channel is still wrong.** Every arm here forecasts consumption with a real model,
  so what a pair measures is the value of the roof *given that load model*. It is not "what a
  perfect forecast is worth" — that is `milp_full`, already in both track notebooks — and the
  two do not add up, because the MILP's response to an error in one channel depends on what it
  believes about the other.

* **Five load models, and they are not a sample.** `hbd` and `hbd_baseline` have no perfect
  twin in the sweep, so the pooled read-out in section 7 is over the five that do. Those five
  span the roster's range on the error axis — from copying yesterday to the best method in the
  study — but a median over five hand-picked models has no interval worth quoting and none is
  printed.

* **The ladder pins the load channel at Prophet, which is not the best load model available.**
  `median14` and `hbd_median14` both beat it on consumption skill. The ladder is therefore the
  value of the roof *to Prophet*; the pair table is the same question asked of every load model
  and is the one to read if the interest is in the best of them.

* **One year, one battery, one pack price.** Every caveat in `CODE_PV_FORECAST.ipynb` applies
  here unchanged — the in-sample rule tuning, the unconverted wear rate on AU, the endogenous
  SI contract. They cancel out of a *paired difference* to the extent that both halves of a
  pair carry them identically, which is exactly, since the two arms differ in one channel of
  one forecast. That is the main argument for reading this notebook in differences rather than
  in levels.


## 8. Putting the figures in the article

Same rule as the other two notebooks: every figure is authored at `\textwidth` (7.16 in), so a
single-column float scales it to 49 % and halves every font in it. The cell below emits the
`figure*` blocks, with each figure's recorded caption already filled in.


In [ ]:
### The LaTeX blocks for this notebook's figures
import json
from pathlib import Path

FIG_DIR = Path("..") / "Results" / "Figures" / SUBDIR
_blocks = []
for _d in sorted(p for p in FIG_DIR.iterdir() if p.is_dir()):
    _src = _d / "source.json"
    if not _src.is_file():
        continue
    _rec = json.loads(_src.read_text())
    # `finish` and `chart_frame` record their titles here because
    # `pf.use(titles=False)` keeps them out of the image, so the caption the
    # article should carry is already written and retyping it is how the two
    # drift apart.
    _cap = " ".join(_rec.get("caption") or []) or _d.name.replace("_", " ")
    _w = _rec["export"]["figsize_in"][0]
    _blocks.append((_d.name, _w, pf.latex_figure(_d.name, _cap, subdir=SUBDIR,
                                                 span=True)))

print(f"{len(_blocks)} figures, all at \\textwidth "
      f"({min(w for _, w, _ in _blocks):.2f}-{max(w for _, w, _ in _blocks):.2f} in)\n"
      if _blocks else "No figures exported yet.\n")
for _name, _, _blk in _blocks:
    print(_blk, "\n")
